# 04 — Compare base / SFT / GRPO on the held-out eval set

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sneh2909/Overfitters/blob/main/notebooks/04_eval_compare.ipynb)
[![W&B](https://img.shields.io/badge/W%26B-house--md-yellow)](https://wandb.ai/sneh2909-christ-university/house-md?nw=nwusersneh2909)

**Goal**: produce the comparison plots that demonstrate **GRPO actually improved over SFT and the base model**. We score every checkpoint on the same fixed 45-patient eval split (`data/eval_set.jsonl`).

Three modes:
* **Fast (default, ~5 sec)**: load pre-computed eval JSONs from the repo and the `SnehShah/house-md-results` dataset, then plot.
* **Live mini-eval (~15 min on T4)**: section 5 — take any LoRA adapter and score 6 patients live against the OpenEnv Space.
* **Full re-run (~30 min on L4 per model)**: section 6 — invoke `scripts/eval.sh` against HF Jobs.

## 1. Install + auth

In [ ]:
%%capture
!pip install -q huggingface_hub pandas matplotlib requests
!pip install -q "openenv-core>=0.2.2"

In [ ]:
import os, json
from pathlib import Path

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

HF_USERNAME  = os.environ.get("HF_USERNAME", "SnehShah")
HF_TOKEN     = os.environ.get("HF_TOKEN", "")
RESULTS_REPO = f"{HF_USERNAME}/house-md-results"

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

## 2. Pull every available eval JSON

We try **three sources in order** for each model tag:
1. The local `results/` dir (cloned with the repo).
2. The **private** `SnehShah/house-md-results` HF dataset.
3. (For random / greedy) reduce them on-the-fly from `results/eval_results.json`.

Anything that's missing is silently skipped — the rest of the notebook still produces a plot from whatever we did find.

In [ ]:
from huggingface_hub import hf_hub_download

TAGS = ["base", "sft", "grpo", "gemini_flash"]
summaries = {}

for tag in TAGS:
    fname = f"eval_{tag}.json" if tag != "gemini_flash" else "gemini_flash.json"
    local = Path("results") / fname
    payload = None
    
    if local.exists():
        payload = json.loads(local.read_text())
        src = f"local: {local}"
    else:
        try:
            p = hf_hub_download(RESULTS_REPO, fname, repo_type="dataset", token=HF_TOKEN or None)
            payload = json.loads(Path(p).read_text())
            src = f"hub: {RESULTS_REPO}/{fname}"
        except Exception as e:
            src = f"missing ({type(e).__name__})"
    
    if payload:
        summaries[tag] = payload["summary"]
        print(f"✓ {tag:<14} — acc {payload['summary']['correct_pct']}%  ({src})")
    else:
        print(f"· {tag:<14} — {src}")

In [ ]:
# Reduce the random / greedy baselines on the fly (no GPU + no model needed).
import statistics

def reduce_baseline(rows):
    n = len(rows)
    if n == 0:
        return None
    correct = sum(1 for r in rows if r["correct"])
    keys = ["r1_accuracy", "r2_cost", "r6_anchoring", "r7_safety", "r8_format", "total"]
    avg = {k: round(statistics.mean(r["rewards"][k] for r in rows), 4) for k in keys}
    return {
        "n_evaluated": n,
        "correct": correct,
        "correct_pct": round(100 * correct / n, 1),
        "avg_rewards": avg,
        "avg_steps": round(statistics.mean(r["steps"] for r in rows), 1),
        "avg_cost": round(statistics.mean(r["cost"] for r in rows), 1),
    }

p = Path("results/eval_results.json")
if p.exists():
    raw = json.loads(p.read_text())
    if isinstance(raw.get("random"), list):
        summaries["random"] = reduce_baseline(raw["random"])
        print(f"✓ random   — acc {summaries['random']['correct_pct']}%")
    if isinstance(raw.get("greedy"), list):
        summaries["greedy"] = reduce_baseline(raw["greedy"])
        print(f"✓ greedy   — acc {summaries['greedy']['correct_pct']}%")

## 3. Comparison table

In [ ]:
import pandas as pd

ORDER = ["random", "greedy", "base", "sft", "grpo", "gemini_flash"]
rows = []
for tag in ORDER:
    s = summaries.get(tag)
    if not s:
        continue
    rows.append({
        "model": tag,
        "n": s["n_evaluated"],
        "acc %": s["correct_pct"],
        "avg total": round(s["avg_rewards"]["total"], 3),
        "r1 accuracy": round(s["avg_rewards"]["r1_accuracy"], 3),
        "r2 cost": round(s["avg_rewards"]["r2_cost"], 3),
        "r6 anchor": round(s["avg_rewards"]["r6_anchoring"], 3),
        "r7 safety": round(s["avg_rewards"]["r7_safety"], 3),
        "r8 format": round(s["avg_rewards"]["r8_format"], 3),
        "avg steps": s.get("avg_steps"),
        "avg cost ($)": s.get("avg_cost"),
    })
df = pd.DataFrame(rows)
df

## 4. Comparison bar chart

Three sub-plots:
* **Accuracy** — fraction of patients we diagnosed correctly.
* **Avg total reward** — the scalar GRPO actually optimises.
* **Avg cost** — the trade-off everyone forgets about.

Compare against the **oracle ceiling**: avg total reward 2.5 – 3.0, cost ~$200, accuracy ~100% by construction.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].bar(df["model"], df["acc %"], color="steelblue")
axes[0].set_title("Accuracy on 45 held-out patients")
axes[0].set_ylabel("correct %")
axes[0].axhline(100, linestyle="--", color="green", alpha=0.6, label="oracle ceiling")
axes[0].legend()

axes[1].bar(df["model"], df["avg total"], color="darkorange")
axes[1].set_title("Avg total reward (5-rubric weighted)")
axes[1].axhline(2.5, linestyle="--", color="green", alpha=0.6, label="oracle ceiling ≈ 2.5")
axes[1].legend()

axes[2].bar(df["model"], df["avg cost ($)"], color="firebrick")
axes[2].set_title("Avg cost per episode ($)")

for ax in axes:
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("eval_compare.png", dpi=120)
plt.show()

### What to look for

* **base → SFT**: r8_format goes from ~0.6 to ~0.95 (the JSON-malformedness rate drops). Accuracy may **dip** because SFT teaches sequencing but doesn't yet reason about *which* test to order.
* **SFT → GRPO**: r1_accuracy and total go up, r2_cost stays roughly flat (cost penalty caps wasteful workups), r7_safety improves on critical-acute cases.
* **gemini_flash** is included as an upper bound from a much larger frontier model with no fine-tuning — it's allowed to peek at no extra ground truth, just smarter prior.

> The canonical version of this comparison plot for the README lives at [`docs/plots/eval_comparison.png`](../docs/plots/eval_comparison.png) and is regenerated by [`scripts/build_training_plots.py`](../scripts/build_training_plots.py). If you change the bar chart above, also re-run that script so the README and the notebook stay in agreement.

## 5. Live mini-eval (optional, requires GPU)

Run any LoRA adapter for 6 patients against the **live OpenEnv Space**. Useful for verifying claims when the eval JSON is missing or stale.

Set `ADAPTER_PATH = ""` to evaluate the **base** Gemma-3-4B-IT directly with no adapter.

In [ ]:
ADAPTER_PATH = "SnehShah/house-md-grpo-optimized-gemma3-4b-v3"
ENV_URL      = "https://snehshah-house-md-env.hf.space"
N_PATIENTS   = 6

import torch
import requests
if not torch.cuda.is_available():
    print("⚠  no GPU — skip this section, the plots above are sufficient")
else:
    from unsloth import FastLanguageModel
    from peft import PeftModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
        max_seq_length=4096, dtype=None, load_in_4bit=True, token=HF_TOKEN or None,
    )
    if ADAPTER_PATH:
        model = PeftModel.from_pretrained(model, ADAPTER_PATH, token=HF_TOKEN or None)
    FastLanguageModel.for_inference(model)
    
    correct = 0; total_r = 0.0
    for i in range(N_PATIENTS):
        s = requests.post(f"{ENV_URL}/api/episodes", json={}).json()
        sid = s["session_id"]
        for _ in range(12):
            ep = requests.get(f"{ENV_URL}/api/episodes/{sid}").json()
            if ep["observation"]["done"]:
                break
            inputs = tokenizer.apply_chat_template(
                [{"role": "user", "content": ep["observation"]["prompt"]}],
                return_tensors="pt", add_generation_prompt=True,
            ).to(model.device)
            with torch.inference_mode():
                out = model.generate(inputs, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.eos_token_id)
            raw = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
            raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            try:
                action = json.loads(raw)
            except Exception:
                action = {"type": "DIAGNOSE", "argument": "viral_uri", "rationale": "parse-fallback"}
            try:
                requests.post(f"{ENV_URL}/api/episodes/{sid}/actions", json={"action": action}, timeout=15)
            except Exception:
                break
        rewards = requests.get(f"{ENV_URL}/api/episodes/{sid}/rewards").json()
        truth = requests.get(f"{ENV_URL}/api/episodes/{sid}/truth").json()
        ep = requests.get(f"{ENV_URL}/api/episodes/{sid}").json()
        diag = ep["observation"].get("diagnosis")
        ok = (diag == truth["disease"])
        correct += int(ok); total_r += rewards.get("total", 0.0)
        print(f"[{i+1}/{N_PATIENTS}] truth={truth['disease']:<22} guess={str(diag):<22} reward={rewards.get('total', 0.0):+.2f} {'✓' if ok else '✗'}")
        requests.delete(f"{ENV_URL}/api/episodes/{sid}")
    
    print(f"\nLive mini-eval ({ADAPTER_PATH or 'base'}): {correct}/{N_PATIENTS} correct, avg total {total_r/N_PATIENTS:+.3f}")

## 6. Full re-run on HF Jobs (45 patients, official numbers)

If you want to **regenerate** the comparison JSONs end-to-end on HF L4 GPUs (the same way the production numbers were produced), use the helper shell script:

```bash
export HF_TOKEN=hf_...
export HF_USERNAME=SnehShah
./scripts/eval.sh base
./scripts/eval.sh sft  SnehShah/house-md-sft-gemma3-4b
./scripts/eval.sh grpo SnehShah/house-md-grpo-optimized-gemma3-4b-v3
```

Each call submits a `l4x1` job that:
1. Pulls the env code from `sneh2909/Overfitters`,
2. Boots `uvicorn house_md_env.server.app:app` inside the same container,
3. Loads the model + adapter (4-bit),
4. Plays all 45 eval patients through the OpenEnv WebSocket client,
5. Pushes `eval_<tag>.json` to `SnehShah/house-md-results`.

Runtime per model: ~30 min. Cost: ~\$0.40 each on the L4 flavour.

## 7. Live W&B run

The full GRPO training run is at <https://wandb.ai/sneh2909-christ-university/house-md?nw=nwusersneh2909>.
Useful panels:

* **`grpo/group_mean_reward`** — mean reward per group; should drift up with curriculum boundaries visible at steps 33 and 67.
* **`grpo/r1_accuracy_mean`** — fraction of group rollouts that diagnosed correctly.
* **`grpo/r8_format_mean`** — should stay near 1.0; if it drops below 0.8 the LR is too high.
* **`grpo/kl`** — (old log_prob − new log_prob); >0.5 sustained means PPO clipping isn't keeping the policy near the SFT init.